# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 5: Fine-tuning a Frontier Model

Now we will use OpenAI's API to fine-tune our own private variant of GPT-4.1-nano

In [3]:
# imports

import os
import re
import json
from pathlib import Path
from dotenv import load_dotenv

# Ensure paths work regardless of CWD (workspace root vs notebook dir)
for base in [Path.cwd(), Path.cwd() / "week6" / "community-contributions" / "ebunilo_week_6"]:
    if (base / "price_is_right.ipynb").exists():
        NOTEBOOK_DIR = base
        break
else:
    NOTEBOOK_DIR = Path.cwd()
JSONL_DIR = NOTEBOOK_DIR / "jsonl"
JSONL_DIR.mkdir(parents=True, exist_ok=True)
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate

In [4]:
# environment

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


In [6]:
openai = OpenAI()

# Data size

OpenAI recommends fine-tuning with a small population of 50-100 examples

I'm going to go with 20,000 points.

This cost me $3.42 - you should stick with 100 examples and the cost will be minimal!

In [7]:
# OpenAI recommends fine-tuning with populations of 50-100 examples
# But as our examples are very small, I'm suggesting we go with 100 examples (and 1 epoch)


fine_tune_train = train[:100]
fine_tune_validation = val[:50]

In [8]:
len(fine_tune_train)

100

In [9]:
fine_tune_train[1]

<KiCA JETFAN 1.0 Mini Electric Air Duster Fan Blower Aluminum, 86,000RPM up to 11m/s Wind Speed for Computer Keyboard Electronics Cleaning, Outdoor Hiking, Camping, Air Cushion Inflation, BBQ = $79.0>

# Step 1

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

In [10]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [11]:
messages_for(fine_tune_train[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.'},
 {'role': 'assistant', 'content': '$64.30'}]

In [12]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [13]:
print(make_jsonl(train[:3]))

{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single\u2011piece oil\u2011rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4\" minimum center\u2011to\u2011center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation."}, {"role": "assistant", "content": "$64.30"}]}
{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Mini Electric Air Duster Fan  \nCategory: Electronics  \nBrand: Kica  \nDescription: Ultra\u2011compact 86,000\u202fRPM electric air duster with 11\u202fm/s wind speed for precise cleaning and inflation.  \nDetails: Powered by a 9.99\u202fWh motor, adjustable in four speed levels, it uses three 

In [14]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [15]:
write_jsonl(fine_tune_train, str(JSONL_DIR / "fine_tune_train.jsonl"))

In [16]:
write_jsonl(fine_tune_validation, str(JSONL_DIR / "fine_tune_validation.jsonl"))

In [17]:
with open(JSONL_DIR / "fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [18]:
train_file

FileObject(id='file-Ld8w8ttn4gFH7U6isJjFmY', bytes=55120, created_at=1772858387, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [19]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [20]:
validation_file

FileObject(id='file-E6VSJKu1iy2k5S6RaoE6Bj', bytes=27637, created_at=1772858399, filename='fine_tune_validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

https://platform.openai.com/storage/files/

# Step 2

## And now time to Fine-tune!

In [ ]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)

FineTuningJob(id='ftjob-aE8Pi6CONdHv6MOS5OvAYzfU', created_at=1772858583, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=3, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-dg9qUSL6VSau7akeLmdVGpVv', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-Ld8w8ttn4gFH7U6isJjFmY', validation_file='file-E6VSJKu1iy2k5S6RaoE6Bj', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=3, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)

In [28]:
openai.fine_tuning.jobs.list(limit=1)

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-aE8Pi6CONdHv6MOS5OvAYzfU', created_at=1772858583, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=3, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-dg9qUSL6VSau7akeLmdVGpVv', result_files=[], seed=42, status='queued', trained_tokens=None, training_file='file-Ld8w8ttn4gFH7U6isJjFmY', validation_file='file-E6VSJKu1iy2k5S6RaoE6Bj', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=3, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_i

In [29]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [30]:
job_id

'ftjob-aE8Pi6CONdHv6MOS5OvAYzfU'

In [31]:
openai.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-aE8Pi6CONdHv6MOS5OvAYzfU', created_at=1772858583, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=3, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-dg9qUSL6VSau7akeLmdVGpVv', result_files=[], seed=42, status='running', trained_tokens=None, training_file='file-Ld8w8ttn4gFH7U6isJjFmY', validation_file='file-E6VSJKu1iy2k5S6RaoE6Bj', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=3, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)

In [50]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-GtA1CWm6OEOcAFOH7dPphryp', created_at=1772850413, level='info', message='Evaluating model against our usage policies', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-reyyGyxoPcaOhWRVfDxA4RN3', created_at=1772850413, level='info', message='New fine-tuned model created', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-3uYUe3OhEnuJgZD6P1LYpeWJ', created_at=1772850368, level='info', message='Step 100/100: training loss=1.04, validation loss=2.00, full validation loss=1.24', object='fine_tuning.job.event', data={'step': 100, 'train_loss': 1.0407823324203491, 'valid_loss': 2.001865863800049, 'total_steps': 100, 'full_valid_loss': 1.244896411895752, 'train_mean_token_accuracy': 0.6666666666666666, 'valid_mean_token_accuracy': 0.6666666666666666, 'full_valid_mean_token_accuracy': 0.76}, type='metrics'),
 FineTuningJobEvent(id='ftevent-XTXCb4XcIX5UIGCjE59KRRhf', created_at=177285

https://platform.openai.com/finetune


# Step 3

Test our fine tuned model

In [42]:
# Retrieve a previous fine-tuned job ID

job_id = "ftjob-0SiD3KYzAfZjf0cqzM8hYUra"

In [43]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [44]:
fine_tuned_model_name

'ft:gpt-4.1-nano-2025-04-14:personal:pricer:DGbfPFqm'

In [45]:
# The prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]

In [46]:
# Try this out

test_messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [47]:
# The inference function


def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [48]:
job = openai.fine_tuning.jobs.retrieve(job_id)
print(f"Status: {job.status}")
print(f"Fine-tuned model: {job.fine_tuned_model}")

Status: succeeded
Fine-tuned model: ft:gpt-4.1-nano-2025-04-14:personal:pricer:DGbfPFqm


In [49]:
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))

219.0
$279.00


In [1]:
!{sys.executable} -m pip install "nbformat>=4.2.0"

/bin/bash: {sys.executable}: command not found


In [50]:
evaluate(gpt_4__1_nano_fine_tuned, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$30 $2 $9 $50 $98 $48 $70 $27 $36 $470 $586 $420 $32 $4 $29 $15 $1 $6 $304 $39 $12 $176 $79 $12 $257 $184 $66 $0 $96 $66 $27 $140 $29 $59 $190 $259 $122 $21 $16 $36 $180 $56 $2 $73 $100 $31 $105 $24 $61 $13 $14 $114 $146 $9 $197 $125 $10 $20 $85 $1 $3 $22 $29 $54 $30 $20 $112 $362 $24 $58 $19 $9 $79 $20 $11 $4 $32 $10 $30 $6 $60 $13 $12 $78 $1 $25 $53 $381 $5 $121 $23 $82 $15 $5 $10 $117 $8 $75 $50 $394 $22 $43 $41 $29 $128 $154 $14 $375 $14 $115 $20 $148 $52 $29 $34 $28 $4 $8 $180 $325 $14 $489 $30 $100 $25 $405 $25 $131 $34 $47 $90 $73 $145 $0 $95 $5 $96 $5 $5 $17 $11 $51 $11 $3 $38 $58 $4 $330 $145 $13 $9 $134 $33 $47 $6 $181 $65 $21 $0 $0 $310 $14 $5 $3 $626 $5 $129 $25 $15 $24 $14 $6 $60 $7 $21 $111 $12 $7 $24 $9 $112 $30 $610 $99 $683 $8 $80 $20 $5 $0 $15 $44 $128 $24 $37 $118 $60 $143 $39 $9 

In [ ]:
# 96.58 - mini 200
# 79.29 - mini 2000
# 82.26 - nano 2000
# 67.75 - nano 20,000